# OpenAI Fine-Tuning Demonstration: Medical Question Answering

This notebook provides a step-by-step guide to fine-tuning a cost-effective OpenAI model, `gpt-3.5-turbo-0125`, for a domain-specific task: **Medical Question Answering**.

Fine-tuning allows us to adapt a base model to a specific style, format, or knowledge domain, leading to improved performance and often reduced inference costs compared to using a larger, general-purpose model with complex prompts.

## 1. Setup and Dependencies

In [ ]:
!uv pip install -q openai datasets pandas python-dotenv

Mounted at /content/drive


In [ ]:
# Verify the required files are present
import os
print("Files in current directory:", [f for f in os.listdir(".") if f.endswith((".jsonl", ".csv", ".env"))])

/content/drive/MyDrive/LLM_Course/Testing/Fine Tuning OpenAI
['fine_tuning_plan.md', '.env', 'medical_qa_train.jsonl', 'medical_qa_valid.jsonl', 'openai_fine_tuning_demo.ipynb']


In [ ]:
import os
from dotenv import load_dotenv

# --- API Key Setup ---
# Option 1 — Google Colab: Load API key from Colab Secrets
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Option 2 — Local (VSCode / Jupyter): Load API key from .env file
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [ ]:
import os
import json
import time
import pandas as pd
from openai import OpenAI
from datasets import load_dataset

# --- Configuration ---
# NOTE: Replace 'YOUR_API_KEY' with your actual OpenAI API key or set it as an environment variable
# The environment variable OPENAI_API_KEY is automatically used by the OpenAI client.
# NOTE: For the fine-tuning steps (3.1 - 3.3) to work, you must have a valid OpenAI API key with billing enabled.
client = OpenAI()

BASE_MODEL = "gpt-3.5-turbo-0125"
DATASET_NAME = "Mohammed-Altaf/medical-instruction-120k"
TRAIN_FILE = "medical_qa_train.jsonl"
VALID_FILE = "medical_qa_valid.jsonl"
NUM_EXAMPLES = 100 # Using a subset for a quick, cost-effective demo
VALID_SPLIT_SIZE = 0.1 # 10% for validation

print("Setup complete.")

Setup complete.


## 2. Data Preparation

We will use the `Detsutut/MedInstruct` dataset from Hugging Face, which contains medical instruction-response pairs. The data needs to be converted into the **OpenAI fine-tuning format**, which is a JSONL file where each line is a dictionary with a `messages` array, following the chat completion format (`system`, `user`, `assistant` roles).

In [ ]:
import re

def convert_hf_conversation_to_openai(sample, system_prompt=None):
    """
    Converts a Hugging Face-style conversation sample to OpenAI fine-tuning format.
    :param sample: dict with 'Conversation' key
    :param system_prompt: Optional string to set as system-level context
    :return: dict in OpenAI format, e.g. {"messages": [ ... ]}
    """
    convo = sample['Conversation']
    # Find all segments [|Human|] <content> or [|AI|] <content>
    pattern = r'\[\|Human\|\]([\s\S]*?)(?=\[\|AI\|\]|\Z)|\[\|AI\|\]([\s\S]*?)(?=\[\|Human\|\]|\Z)'
    segments = re.findall(pattern, convo)
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    for human, ai in segments:
        if human:
            messages.append({"role": "user", "content": human.strip()})
        if ai:
            messages.append({"role": "assistant", "content": ai.strip()})
    return {"messages": messages}


print(f"Loading dataset: {DATASET_NAME}...")
dataset = load_dataset(DATASET_NAME, split="train")

# Take a subset for the demo (5000 examples) to keep the fine-tuning cost low and the demo fast.
# NOTE: The dataset structure was corrected from 'instruction'/'response' to 'instruction'/'output' based on inspection.
dataset_subset = dataset.shuffle(seed=42).select(range(NUM_EXAMPLES))

# Apply the formatting function
formatted_data = [convert_hf_conversation_to_openai(sample) for sample in dataset_subset]

# Split into training and validation sets
valid_size = int(NUM_EXAMPLES * VALID_SPLIT_SIZE)
train_data = formatted_data[valid_size:]
valid_data = formatted_data[:valid_size]

print(f"Total examples: {len(formatted_data)}")
print(f"Training examples: {len(train_data)}")
print(f"Validation examples: {len(valid_data)}")

# Save to JSONL files
def save_to_jsonl(data, filename):
    with open(filename, 'w', encoding='utf-8') as f:
        for entry in data:
            f.write(json.dumps(entry) + '\n')

save_to_jsonl(train_data, TRAIN_FILE)
save_to_jsonl(valid_data, VALID_FILE)

print(f"Data saved to {TRAIN_FILE} and {VALID_FILE}.")
print("Example of a formatted training entry:")
print(json.dumps(train_data[0], indent=2))

Loading dataset: Mohammed-Altaf/medical-instruction-120k...
Total examples: 100
Training examples: 90
Validation examples: 10
Data saved to medical_qa_train.jsonl and medical_qa_valid.jsonl.
Example of a formatted training entry:
{
  "messages": [
    {
      "role": "user",
      "content": "my 10year old child had done peripheral smear last year when he was suffering from viral fever, it shows hypochromic megaloblastic anemia with active lymphocytes.his hb count is only 8.4.now his hb is 12.2 .he advised me to take bone marrow test.but i dis agree.is this smear shows any serious problem"
    },
    {
      "role": "assistant",
      "content": "Megaloblastic anemia can be due to a deficiency in a vitamin, B12.  Hypo chromic just means low iron content.  The only information here for his current test is his HGB which appears to have recovered after the illness which caused it to decrease earlier.  Some viral illnesses can slow down red cell production, but recovery is spontaneous.  To

In [ ]:
dataset_subset[0]

{'Conversation': 'The conversation between human and AI assistant.\n[|Human|] I am a teen and I am extremely territorial. When people come into my room when I don t want them to, I get agitated and anxious very quickly. But when people come into my room and move my things... it sends me into a rage. My heart starts racing, I begin to shake and I start crying and screaming for that person to leave. It doesn t happen a ton, but the first time I remember it, I was 8. As a kid I was very competitive and would get angry really quickly if I lost.\n[|AI|]  Dear serve understand your concerns went through your details. You must introspect. Do you really have this anti-social behavior pattern with everyone who interacts with you ? Or is this behavior selective ? Do you become angry with everyone who comes to your room ? Or are you selective ? Do you scream at ever occasion or scream only when you are stressed ? The answer to these questions helps us to identify whether you are actually anti-soc

In [ ]:
# before
dataset_subset[-1]

{'Conversation': "The conversation between human and AI assistant.\n[|Human|] My husband had severe pain in his chest and shoulder area, he thought it was a heart attack. That was ruled out at the hospital. Also the gall bladder was tested. Many x-rays, MRI s and test did not reveal anything. If he does much bouncing operating equipment, it will come on sometimes. He is a diabetic and on many medications including gabapentin 100 MG/ Doctors says he a mystery ? What could it be ?\n[|AI|]  Hi, thank-you for providing the brief history. Well, it is one way good to hear that your husband do not have any systemic pathology. And all thanks to God. Now since his pain is the symptom which is worrying him, it will be more or w muscular. Pectorals muscle takes its origin from the center of the chest up to the shoulder bone. So if the muscle is having a pain it cannot be traced by a diagnostic tests performed. Also to get rid of this pain, the only thing to try as per me is slow gentle massage, t

In [ ]:
# after formatting
print(json.dumps(train_data[-1], indent=2))

{
  "messages": [
    {
      "role": "user",
      "content": "My husband had severe pain in his chest and shoulder area, he thought it was a heart attack. That was ruled out at the hospital. Also the gall bladder was tested. Many x-rays, MRI s and test did not reveal anything. If he does much bouncing operating equipment, it will come on sometimes. He is a diabetic and on many medications including gabapentin 100 MG/ Doctors says he a mystery ? What could it be ?"
    },
    {
      "role": "assistant",
      "content": "Hi, thank-you for providing the brief history. Well, it is one way good to hear that your husband do not have any systemic pathology. And all thanks to God. Now since his pain is the symptom which is worrying him, it will be more or w muscular. Pectorals muscle takes its origin from the center of the chest up to the shoulder bone. So if the muscle is having a pain it cannot be traced by a diagnostic tests performed. Also to get rid of this pain, the only thing to try

## 3. Fine-Tuning Process

The fine-tuning process involves three main steps:
1. Uploading the training and validation files.
2. Creating the fine-tuning job.
3. Monitoring the job until completion.

In [ ]:
# 3.1 Upload Files
print(f"Uploading training file: {TRAIN_FILE}...")
training_file = client.files.create(
    file=open(TRAIN_FILE, "rb"),
    purpose="fine-tune"
)
print(f"Training File ID: {training_file.id}")

print(f"Uploading validation file: {VALID_FILE}...")
validation_file = client.files.create(
    file=open(VALID_FILE, "rb"),
    purpose="fine-tune"
)
print(f"Validation File ID: {validation_file.id}")

Uploading training file: medical_qa_train.jsonl...
Training File ID: file-CrF9uDPeChDDQA79hdV3qy
Uploading validation file: medical_qa_valid.jsonl...
Validation File ID: file-WogJGLaGHPEKyTKDUqheN9


In [ ]:
# 3.2 Create Fine-Tuning Job
print(f"Creating fine-tuning job on base model {BASE_MODEL}...")
fine_tuning_job = client.fine_tuning.jobs.create(
    training_file=training_file.id,
    validation_file=validation_file.id,
    model=BASE_MODEL
)

JOB_ID = fine_tuning_job.id
print(f"Fine-Tuning Job ID: {JOB_ID}")
print("Job status: ", fine_tuning_job.status)

Creating fine-tuning job on base model gpt-3.5-turbo-0125...
Fine-Tuning Job ID: ftjob-9WCBieYgUtEkIlWB7RRxKLoR
Job status:  validating_files


In [ ]:
fine_tuning_job = client.fine_tuning.jobs.retrieve('ftjob-9WCBieYgUtEkIlWB7RRxKLoR')
print(fine_tuning_job.status)  # should be 'succeeded'

succeeded


In [ ]:
FINE_TUNED_MODEL = fine_tuning_job.fine_tuned_model
print(FINE_TUNED_MODEL)

ft:gpt-3.5-turbo-0125:personal::Cf6fKcm9


In [ ]:
# 3.3 Monitor Job Status
print("Monitoring job status. This may take a while (hours)...")
# NOTE: The fine-tuning job cannot be executed in this environment.
# You must run the file upload and job creation steps in your local environment with a valid API key.
if fine_tuning_job.status == "succeeded":
    FINE_TUNED_MODEL = fine_tuning_job.fine_tuned_model
    print("\nFine-tuning job succeeded!")
    print(f"Fine-Tuned Model Name: {FINE_TUNED_MODEL}")
else:
    print(f"\nFine-tuning job failed or was cancelled. Status: {fine_tuning_job.status}")
    FINE_TUNED_MODEL = None

Monitoring job status. This may take a while (hours)...

Fine-tuning job succeeded!
Fine-Tuned Model Name: ft:gpt-3.5-turbo-0125:personal::Cf6fKcm9


## 4. Evaluation and Comparison

Once the model is fine-tuned, we can compare its performance against the base model using a sample question from the validation set.

In [ ]:
test_messages[1]

{'role': 'assistant',
 'content': 'Dear serve understand your concerns went through your details. You must introspect. Do you really have this anti-social behavior pattern with everyone who interacts with you ? Or is this behavior selective ? Do you become angry with everyone who comes to your room ? Or are you selective ? Do you scream at ever occasion or scream only when you are stressed ? The answer to these questions helps us to identify whether you are actually anti-social or not. You are a teen. Naturally you will have the lot of physical, organic and hormonal changes happening inside you. Almost all teens are aggressive towards their parents and teachers. That is natural. Provide all the answers to the above questions. Let us clear your problem. In the meantime, do not worry. If you require more of my help in this aspect, please use this URL. http://goo.gl/aYW2pR. Make sure that you include every minute details possible. Hope this answers your query. Available for further clarif

In [ ]:
if FINE_TUNED_MODEL:
    # Get a test question from the validation set
    test_entry = valid_data[0]
    test_messages = test_entry["messages"]
    test_question = test_messages[0]["content"]
    expected_answer = test_messages[1]["content"]

    print(f"--- Test Question ---\n{test_question}\n")
    print(f"--- Expected Answer (from dataset) ---\n{expected_answer}\n")

    def get_completion(model, messages):
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0
        )
        return response.choices[0].message.content

    # 4.1 Test Base Model
    print("--- Base Model Response ---")
    base_messages = [
        {"role": "system", "content": "You are a helpful and accurate medical question-answering assistant. Provide concise and medically sound responses."},
        {"role": "user", "content": test_question}
    ]
    base_response = get_completion(BASE_MODEL, base_messages)
    print(base_response)

    # 4.2 Test Fine-Tuned Model
    print("\n--- Fine-Tuned Model Response ---")
    # Use the fine-tuned model name
    ft_response = get_completion(FINE_TUNED_MODEL, base_messages)
    print(ft_response)

    print("\nComparison complete. The fine-tuned model should exhibit a more domain-specific and style-adherent response.")
else:
    print("Cannot run evaluation as the fine-tuning job did not succeed.")

--- Test Question ---
I am a teen and I am extremely territorial. When people come into my room when I don t want them to, I get agitated and anxious very quickly. But when people come into my room and move my things... it sends me into a rage. My heart starts racing, I begin to shake and I start crying and screaming for that person to leave. It doesn t happen a ton, but the first time I remember it, I was 8. As a kid I was very competitive and would get angry really quickly if I lost.

--- Expected Answer (from dataset) ---
Dear serve understand your concerns went through your details. You must introspect. Do you really have this anti-social behavior pattern with everyone who interacts with you ? Or is this behavior selective ? Do you become angry with everyone who comes to your room ? Or are you selective ? Do you scream at ever occasion or scream only when you are stressed ? The answer to these questions helps us to identify whether you are actually anti-social or not. You are a tee

## 5. Cleanup (Optional but Recommended)

To avoid unnecessary storage costs, it is recommended to delete the uploaded files and the fine-tuned model after you are done with the demonstration.

In [ ]:
if FINE_TUNED_MODEL:
    # Delete the fine-tuned model
    try:
        client.models.delete(FINE_TUNED_MODEL)
        print(f"Successfully deleted fine-tuned model: {FINE_TUNED_MODEL}")
    except Exception as e:
        print(f"Could not delete model {FINE_TUNED_MODEL}. Error: {e}")

# Delete the uploaded files
try:
    # NOTE: This cleanup step is crucial to avoid recurring costs for the fine-tuned model and file storage.
    # If you run the fine-tuning job locally, make sure to run the following commands to clean up:
    # client.files.delete(training_file.id)
    # client.files.delete(validation_file.id)
    print("Cleanup code is commented out for sandbox environment.")
except Exception as e:
    print(f"Could not delete uploaded files. Error: {e}")